# Chapter 22: FastSLAM

<a href="../lite/lab/index.html?path=ch22_fastslam.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import time

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def draw_cov_ellipse(ax, mean, cov, n_std=2, **kwargs):
    from matplotlib.patches import Ellipse
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    ax.add_patch(Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs))

EKF-SLAM chokes on large maps because every landmark is correlated with every
other. FastSLAM exploits a clever factorization: if you knew the robot's exact
path, every landmark would be independent. So it samples many possible paths
(particles), and each path maintains its own tiny independent landmark filters.
A thousand parallel SLAM systems, each one cheap.

This chapter implements **FastSLAM 1.0**, explains why the factorization works,
and compares it with EKF-SLAM on both small and large maps.

## 22.1 Problem Factorization: Rao-Blackwellization

The SLAM posterior can be factored as:

$$p(\mathbf{x}_{1:t}, \mathbf{m} \mid \mathbf{z}_{1:t}, \mathbf{u}_{1:t})
= p(\mathbf{x}_{1:t} \mid \mathbf{z}_{1:t}, \mathbf{u}_{1:t})
\prod_{j=1}^{N} p(\mathbf{m}_j \mid \mathbf{x}_{1:t}, \mathbf{z}_{1:t})$$

The key insight: **conditioned on the robot path**, each landmark is independent.
This means we can:
1. Sample robot paths using a **particle filter**
2. For each particle, maintain **independent** 2x2 EKFs for each landmark

No giant covariance matrix. No quadratic scaling.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_landmarks_demo = 5
# ──────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# EKF-SLAM: one big covariance
ax = axes[0]
dim_ekf = 3 + 2 * n_landmarks_demo
P_ekf = np.random.randn(dim_ekf, dim_ekf)
P_ekf = P_ekf @ P_ekf.T + np.eye(dim_ekf)  # make positive definite
ax.imshow(np.abs(P_ekf), cmap='Blues', interpolation='nearest')
ax.set_title(f'EKF-SLAM: one {dim_ekf}x{dim_ekf} covariance\n(everything coupled)', fontsize=11)
ax.set_xlabel('State index')

# FastSLAM: many independent small covariances
ax = axes[1]
P_fast = np.zeros((dim_ekf, dim_ekf))
P_fast[:3, :3] = 0  # no robot covariance (sampled)
for i in range(n_landmarks_demo):
    idx = 3 + 2*i
    block = np.random.randn(2, 2)
    P_fast[idx:idx+2, idx:idx+2] = block @ block.T + np.eye(2)
ax.imshow(np.abs(P_fast), cmap='Oranges', interpolation='nearest')
ax.set_title(f'FastSLAM: {n_landmarks_demo} independent 2x2 blocks\n(no cross-correlations)', fontsize=11)
ax.set_xlabel('State index')

# Storage comparison
ax = axes[2]
ns = np.arange(5, 201, 5)
ekf_storage = (3 + 2*ns)**2
fast_storage = 50 * (3 + ns * 4)  # 50 particles, each: 3 pose + N*(2+2x2) per landmark
ax.semilogy(ns, ekf_storage, 'steelblue', lw=2, label='EKF-SLAM')
ax.semilogy(ns, fast_storage, 'orange', lw=2, label='FastSLAM (50 particles)')
ax.set_xlabel('Number of landmarks')
ax.set_ylabel('Storage (floats)')
ax.set_title('Storage scaling', fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

print(f'EKF-SLAM at {n_landmarks_demo} landmarks: {dim_ekf**2} covariance entries')
print(f'FastSLAM at {n_landmarks_demo} landmarks: {n_landmarks_demo * 4} per particle (2x2 blocks only)')

**Key insight:** The Rao-Blackwellization splits SLAM into two parts. The robot
path is handled by sampling (particle filter). The map, conditioned on each
sampled path, decomposes into independent landmark filters. This eliminates the
cross-covariance blocks that make EKF-SLAM expensive.

## 22.2 Particle Trajectory: Each Particle = One Possible Robot Path

Each particle $k$ carries:
- A robot pose $\mathbf{x}_t^{[k]}$ (sampled from the motion model)
- A weight $w^{[k]}$ (how well its observations match)
- An importance weight used for resampling

During the prediction step, each particle propagates independently through the
motion model with sampled noise.

In [ ]:
def motion_model(pose, u, noise_std, dt=1.0):
    """Move a pose with velocity command u=[v, omega] plus noise."""
    v = u[0] + np.random.randn() * noise_std[0]
    omega = u[1] + np.random.randn() * noise_std[1]
    x, y, theta = pose
    if abs(omega) < 1e-6:
        x_new = x + v * dt * np.cos(theta)
        y_new = y + v * dt * np.sin(theta)
        theta_new = theta
    else:
        x_new = x + v/omega * (np.sin(theta + omega*dt) - np.sin(theta))
        y_new = y + v/omega * (np.cos(theta) - np.cos(theta + omega*dt))
        theta_new = theta + omega * dt
    return np.array([x_new, y_new, (theta_new + np.pi) % (2*np.pi) - np.pi])

def observation_model(robot, landmark):
    """Range-bearing observation."""
    dx = landmark[0] - robot[0]
    dy = landmark[1] - robot[1]
    r = np.sqrt(dx**2 + dy**2)
    phi = np.arctan2(dy, dx) - robot[2]
    phi = (phi + np.pi) % (2*np.pi) - np.pi
    return np.array([r, phi])

print('Motion and observation models defined.')

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_particles = 50
n_steps_demo = 20
sigma_v = 0.2
sigma_omega = 0.05
# ──────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
poses = np.tile([0.0, 0.0, 0.0], (n_particles, 1))
u_cmd = [1.0, 0.15]  # drive forward, turning slightly

trajectories = np.zeros((n_particles, n_steps_demo + 1, 3))
trajectories[:, 0, :] = poses

true_pose = np.array([0.0, 0.0, 0.0])
true_traj = [true_pose.copy()]

for t in range(n_steps_demo):
    for k in range(n_particles):
        poses[k] = motion_model(poses[k], u_cmd, [sigma_v, sigma_omega])
        trajectories[k, t+1, :] = poses[k]
    true_pose = motion_model(true_pose, u_cmd, [0, 0])  # no noise
    true_traj.append(true_pose.copy())

true_traj = np.array(true_traj)

fig, ax = plt.subplots(figsize=(10, 7))
for k in range(n_particles):
    ax.plot(trajectories[k, :, 0], trajectories[k, :, 1],
            'steelblue', alpha=0.2, lw=0.8)
ax.plot(true_traj[:, 0], true_traj[:, 1], 'k-', lw=3, label='True path')
ax.scatter(poses[:, 0], poses[:, 1], c='tomato', s=20, zorder=5, label='Particles (final)')
ax.set_aspect('equal')
ax.set_title(f'{n_particles} particle trajectories (each = one possible robot path)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

**Observation:** Each blue line is one particle's trajectory. The spread of
particles represents the robot's pose uncertainty. Without observations to
weight and resample, the cloud diverges over time.

## 22.3 Landmark Estimation: Per-Particle EKF

Each particle maintains an **independent** 2D EKF for each landmark:

- Mean $\boldsymbol{\mu}_j^{[k]}$ (2D position of landmark $j$ as estimated by particle $k$)
- Covariance $\Sigma_j^{[k]}$ ($2 \times 2$ matrix)

When particle $k$ observes landmark $j$, it runs a standard 2D EKF update
using its own pose $\mathbf{x}^{[k]}$ as the robot location. Since the pose is
"known" (for that particle), the landmark is independent of all others.

The particle weight is updated based on the observation likelihood:

$$w^{[k]} \propto \det(2\pi S_j^{[k]})^{-1/2} \exp\left(-\tfrac{1}{2} \boldsymbol{\nu}^T (S_j^{[k]})^{-1} \boldsymbol{\nu}\right)$$

where $\boldsymbol{\nu}$ is the innovation and $S_j^{[k]} = H_j \Sigma_j^{[k]} H_j^T + R$.

In [ ]:
class Particle:
    """A single FastSLAM particle."""
    def __init__(self, pose, n_landmarks):
        self.pose = pose.copy()
        self.weight = 1.0
        # Per-landmark EKFs
        self.lm_mean = [None] * n_landmarks
        self.lm_cov = [None] * n_landmarks
        self.lm_seen = [False] * n_landmarks
    
    def predict(self, u, noise_std):
        self.pose = motion_model(self.pose, u, noise_std)
    
    def update(self, lm_idx, z, R):
        """Update landmark estimate given observation z=[range, bearing]."""
        if not self.lm_seen[lm_idx]:
            # Initialize landmark from first observation
            r, phi = z
            angle = phi + self.pose[2]
            lm_x = self.pose[0] + r * np.cos(angle)
            lm_y = self.pose[1] + r * np.sin(angle)
            self.lm_mean[lm_idx] = np.array([lm_x, lm_y])
            # Initial covariance: large
            self.lm_cov[lm_idx] = np.eye(2) * 10.0
            self.lm_seen[lm_idx] = True
            return
        
        # Predict observation
        mu = self.lm_mean[lm_idx]
        Sigma = self.lm_cov[lm_idx]
        z_pred = observation_model(self.pose, mu)
        
        # Jacobian w.r.t. landmark only (2x2)
        dx = mu[0] - self.pose[0]
        dy = mu[1] - self.pose[1]
        q = dx**2 + dy**2
        r_pred = np.sqrt(q)
        H = np.array([[dx/r_pred, dy/r_pred],
                       [-dy/q, dx/q]])
        
        # Innovation
        innov = z - z_pred
        innov[1] = (innov[1] + np.pi) % (2*np.pi) - np.pi
        
        S = H @ Sigma @ H.T + R
        K = Sigma @ H.T @ np.linalg.inv(S)
        
        self.lm_mean[lm_idx] = mu + K @ innov
        self.lm_cov[lm_idx] = (np.eye(2) - K @ H) @ Sigma
        
        # Update weight (observation likelihood)
        det_S = max(np.linalg.det(S), 1e-10)
        self.weight *= (1.0 / np.sqrt((2*np.pi)**2 * det_S)) * \
            np.exp(-0.5 * innov @ np.linalg.inv(S) @ innov)

print('Particle class defined with predict and update methods.')

In [ ]:
def resample(particles):
    """Low-variance resampling."""
    import copy
    N = len(particles)
    weights = np.array([p.weight for p in particles])
    w_sum = weights.sum()
    if w_sum < 1e-20:
        weights = np.ones(N) / N
    else:
        weights /= w_sum
    
    # Low-variance resampling
    new_particles = []
    r = np.random.uniform(0, 1.0/N)
    c = weights[0]
    i = 0
    for j in range(N):
        u_val = r + j / N
        while c < u_val and i < N - 1:
            i += 1
            c += weights[i]
        new_p = copy.deepcopy(particles[i])
        new_p.weight = 1.0 / N
        new_particles.append(new_p)
    return new_particles

print('Low-variance resampling function defined.')

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
N_P = 30              # number of particles
N_LM_small = 5        # landmarks
sigma_v_fs = 0.15
sigma_om_fs = 0.05
sigma_r_fs = 0.3
sigma_b_fs = 0.1
max_range_fs = 6.0
# ──────────────────────────────────────────────────────────────────────────────

# Setup
lm_true = np.array([[3, 7], [7, 8], [9, 3], [5, 1], [1, 4]], dtype=float)
R_fs = np.diag([sigma_r_fs**2, sigma_b_fs**2])

particles = [Particle(np.array([1.0, 1.0, 0.0]), N_LM_small) for _ in range(N_P)]
true_pose_fs = np.array([1.0, 1.0, 0.0])

# Drive commands for a loop
commands = []
for _ in range(8): commands.append([1.2, 0.0])    # forward
for _ in range(5): commands.append([1.0, 0.35])   # turn left
for _ in range(6): commands.append([1.2, 0.0])    # forward
for _ in range(5): commands.append([1.0, 0.35])   # turn left
for _ in range(8): commands.append([1.2, 0.0])    # forward
for _ in range(5): commands.append([1.0, 0.35])   # turn left
for _ in range(6): commands.append([1.2, 0.0])    # forward

true_path_fs = [true_pose_fs[:2].copy()]

for t, u in enumerate(commands):
    # True motion
    true_pose_fs = motion_model(true_pose_fs, u, [0, 0])
    true_path_fs.append(true_pose_fs[:2].copy())
    
    # Predict all particles
    for p in particles:
        p.predict(u, [sigma_v_fs, sigma_om_fs])
    
    # Generate observations for landmarks in range
    for j in range(N_LM_small):
        dx = lm_true[j, 0] - true_pose_fs[0]
        dy = lm_true[j, 1] - true_pose_fs[1]
        if np.sqrt(dx**2 + dy**2) > max_range_fs:
            continue
        z_true = observation_model(true_pose_fs, lm_true[j])
        z_noisy = z_true + np.array([np.random.randn()*sigma_r_fs,
                                      np.random.randn()*sigma_b_fs])
        for p in particles:
            p.update(j, z_noisy, R_fs)
    
    # Resample
    particles = resample(particles)

true_path_fs = np.array(true_path_fs)
print(f'Simulation complete: {len(commands)} steps, {N_P} particles, {N_LM_small} landmarks')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

ax = axes[0]
ax.plot(true_path_fs[:, 0], true_path_fs[:, 1], 'k--', lw=2, label='True path')
for p in particles:
    ax.plot(p.pose[0], p.pose[1], '.', color='steelblue', ms=6, alpha=0.5)
ax.scatter(lm_true[:, 0], lm_true[:, 1], c='forestgreen', s=120, marker='*',
           zorder=5, label='True landmarks')

# Show best particle's landmarks
best = max(particles, key=lambda p: p.weight)
for j in range(N_LM_small):
    if best.lm_seen[j]:
        ax.scatter(*best.lm_mean[j], c='tomato', s=50, marker='^', zorder=5)
        draw_cov_ellipse(ax, best.lm_mean[j], best.lm_cov[j],
                         fill=True, facecolor='tomato', alpha=0.2,
                         edgecolor='tomato', lw=1.5)

ax.set_aspect('equal')
ax.set_title('FastSLAM result: particles + best map', fontsize=13)
ax.legend(fontsize=9)

# Per-landmark errors
ax = axes[1]
errors = []
for j in range(N_LM_small):
    if best.lm_seen[j]:
        err = np.linalg.norm(best.lm_mean[j] - lm_true[j])
        errors.append(err)
    else:
        errors.append(np.nan)

ax.bar(range(N_LM_small), errors, color='steelblue', alpha=0.7)
ax.set_xlabel('Landmark index')
ax.set_ylabel('Position error (m)')
ax.set_title('Landmark estimation error (best particle)', fontsize=13)
ax.set_xticks(range(N_LM_small))

plt.tight_layout()
plt.show()

print(f'Mean landmark error: {np.nanmean(errors):.3f} m')

**Observation:** Each particle maintains its own independent landmark estimates.
The best particle (highest weight) typically has the most accurate map. The
landmark covariances are small 2x2 matrices, not part of a giant joint covariance.

## 22.4 Tradeoffs: FastSLAM vs EKF-SLAM

| Property | EKF-SLAM | FastSLAM |
|----------|:--------:|:--------:|
| **Representation** | Single Gaussian | Particle set |
| **Map correlations** | Full $(3+2N)^2$ covariance | Independent per particle |
| **Update cost** | $O(N^2)$ per observation | $O(M \cdot N)$, $M$ = particles |
| **Multimodal** | No | Yes |
| **Particle depletion** | N/A | Can be a problem |
| **Loop closure** | Through cross-covariance | Through resampling |

FastSLAM scales as $O(M \cdot N \log N)$ (with a tree), while EKF-SLAM
scales as $O(N^2)$. For large $N$, FastSLAM wins.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
test_ns = [10, 25, 50, 100, 200]
n_particles_test = 30
n_trials = 3
# ──────────────────────────────────────────────────────────────────────────────

# Measure wall-clock time for EKF-SLAM update vs FastSLAM update
ekf_times = []
fast_times = []

for n_lm in test_ns:
    # EKF-SLAM: update requires O(N^2) operations
    sd = 3 + 2 * n_lm
    x_e = np.random.randn(sd)
    P_e = np.eye(sd)
    H_e = np.zeros((2, sd))
    H_e[0, 0] = -1; H_e[0, 3] = 1
    H_e[1, 1] = -1; H_e[1, 4] = 1
    R_e = np.eye(2) * 0.1
    
    t0 = time.perf_counter()
    for _ in range(n_trials):
        S_e = H_e @ P_e @ H_e.T + R_e
        K_e = P_e @ H_e.T @ np.linalg.inv(S_e)
        P_e_new = (np.eye(sd) - K_e @ H_e) @ P_e
    ekf_times.append((time.perf_counter() - t0) / n_trials * 1000)
    
    # FastSLAM: update requires O(M * 1) per landmark (2x2 EKF)
    t0 = time.perf_counter()
    for _ in range(n_trials):
        for k in range(n_particles_test):
            Sigma_j = np.eye(2)
            H_f = np.eye(2)
            S_f = H_f @ Sigma_j @ H_f.T + R_e
            K_f = Sigma_j @ H_f.T @ np.linalg.inv(S_f)
            Sigma_j = (np.eye(2) - K_f @ H_f) @ Sigma_j
    fast_times.append((time.perf_counter() - t0) / n_trials * 1000)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(test_ns, ekf_times, 'steelblue', lw=2, marker='o', ms=8, label='EKF-SLAM update')
ax.plot(test_ns, fast_times, 'orange', lw=2, marker='s', ms=8, label=f'FastSLAM update ({n_particles_test} particles)')
ax.set_xlabel('Number of landmarks', fontsize=12)
ax.set_ylabel('Time per update (ms)', fontsize=12)
ax.set_title('Update time: EKF-SLAM vs FastSLAM', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

**Observation:** EKF-SLAM update time grows quadratically with landmark count.
FastSLAM update time grows linearly (times the number of particles). The
crossover point depends on the number of particles, but FastSLAM generally
wins for large maps.

---

## Capstone: FastSLAM vs EKF-SLAM on a Larger Map

We compare both algorithms on the same problem: a robot driving a loop past
landmarks. First 10 landmarks (both algorithms should do well), then 100
landmarks (where EKF-SLAM becomes slow).

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(99)
N_LM_cap = 10
N_P_cap = 50
sigma_v_cap = 0.12
sigma_om_cap = 0.04
sigma_r_cap = 0.3
sigma_b_cap = 0.1
max_range_cap = 5.0
# ──────────────────────────────────────────────────────────────────────────────

# Place landmarks in a ring
angles_cap = np.linspace(0, 2*np.pi, N_LM_cap, endpoint=False)
lm_cap = np.column_stack([6*np.cos(angles_cap), 6*np.sin(angles_cap)])

# Drive in a circle
n_steps_cap = 40
drive_r = 3.5
drive_om = 2 * np.pi / n_steps_cap
drive_vel = drive_r * drive_om

R_cap = np.diag([sigma_r_cap**2, sigma_b_cap**2])

# --- FastSLAM ---
fast_particles = [Particle(np.array([drive_r, 0.0, np.pi/2]), N_LM_cap) for _ in range(N_P_cap)]
true_pose_cap = np.array([drive_r, 0.0, np.pi/2])
true_path_cap = [true_pose_cap[:2].copy()]

t0_fast = time.perf_counter()
for t in range(n_steps_cap):
    u_cap = [drive_vel, drive_om]
    true_pose_cap = motion_model(true_pose_cap, u_cap, [0, 0])
    true_path_cap.append(true_pose_cap[:2].copy())
    
    for p in fast_particles:
        p.predict(u_cap, [sigma_v_cap, sigma_om_cap])
    
    for j in range(N_LM_cap):
        dx = lm_cap[j, 0] - true_pose_cap[0]
        dy = lm_cap[j, 1] - true_pose_cap[1]
        if np.sqrt(dx**2 + dy**2) > max_range_cap:
            continue
        z_true = observation_model(true_pose_cap, lm_cap[j])
        z_noisy = z_true + np.array([np.random.randn()*sigma_r_cap,
                                      np.random.randn()*sigma_b_cap])
        for p in fast_particles:
            p.update(j, z_noisy, R_cap)
    
    fast_particles = resample(fast_particles)

t_fast = time.perf_counter() - t0_fast
true_path_cap = np.array(true_path_cap)

best_p = max(fast_particles, key=lambda p: p.weight)

fast_errors = []
for j in range(N_LM_cap):
    if best_p.lm_seen[j]:
        fast_errors.append(np.linalg.norm(best_p.lm_mean[j] - lm_cap[j]))

print(f'FastSLAM: {t_fast:.3f} s, mean error: {np.mean(fast_errors):.3f} m')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

ax = axes[0]
ax.plot(true_path_cap[:, 0], true_path_cap[:, 1], 'k--', lw=2, label='True path')
for p in fast_particles[:10]:  # show subset
    ax.plot(p.pose[0], p.pose[1], '.', color='steelblue', ms=8, alpha=0.5)
ax.scatter(lm_cap[:, 0], lm_cap[:, 1], c='forestgreen', s=120, marker='*',
           zorder=5, label='True landmarks')
for j in range(N_LM_cap):
    if best_p.lm_seen[j]:
        ax.scatter(*best_p.lm_mean[j], c='tomato', s=50, marker='^', zorder=5)
        draw_cov_ellipse(ax, best_p.lm_mean[j], best_p.lm_cov[j],
                         fill=True, facecolor='tomato', alpha=0.2,
                         edgecolor='tomato', lw=1)
ax.set_aspect('equal'); ax.set_title('FastSLAM (10 landmarks)', fontsize=13)
ax.legend(fontsize=9)

ax = axes[1]
ax.bar(range(len(fast_errors)), fast_errors, color='orange', alpha=0.7)
ax.set_xlabel('Landmark index')
ax.set_ylabel('Error (m)')
ax.set_title(f'FastSLAM errors (mean={np.mean(fast_errors):.3f} m)', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
N_LM_large = 100
N_P_large = 30
# ──────────────────────────────────────────────────────────────────────────────

np.random.seed(55)
lm_large = np.random.uniform(-8, 8, (N_LM_large, 2))

# FastSLAM on 100 landmarks
fast_large = [Particle(np.array([drive_r, 0.0, np.pi/2]), N_LM_large) for _ in range(N_P_large)]
true_pose_l = np.array([drive_r, 0.0, np.pi/2])

t0_fast_l = time.perf_counter()
for t in range(n_steps_cap):
    u_cap = [drive_vel, drive_om]
    true_pose_l = motion_model(true_pose_l, u_cap, [0, 0])
    for p in fast_large:
        p.predict(u_cap, [sigma_v_cap, sigma_om_cap])
    for j in range(N_LM_large):
        dx = lm_large[j, 0] - true_pose_l[0]
        dy = lm_large[j, 1] - true_pose_l[1]
        if np.sqrt(dx**2 + dy**2) > max_range_cap:
            continue
        z_true = observation_model(true_pose_l, lm_large[j])
        z_noisy = z_true + np.array([np.random.randn()*sigma_r_cap,
                                      np.random.randn()*sigma_b_cap])
        for p in fast_large:
            p.update(j, z_noisy, R_cap)
    fast_large = resample(fast_large)

t_fast_l = time.perf_counter() - t0_fast_l

# EKF-SLAM equivalent timing estimate for 100 landmarks
sd_100 = 3 + 2 * N_LM_large
t0_ekf_est = time.perf_counter()
P_100 = np.eye(sd_100)
H_100 = np.zeros((2, sd_100))
H_100[0, 0] = -1; H_100[0, 3] = 1; H_100[1, 1] = -1; H_100[1, 4] = 1
for _ in range(n_steps_cap):
    S_100 = H_100 @ P_100 @ H_100.T + R_cap
    K_100 = P_100 @ H_100.T @ np.linalg.inv(S_100)
    P_100 = (np.eye(sd_100) - K_100 @ H_100) @ P_100
t_ekf_est = time.perf_counter() - t0_ekf_est

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['FastSLAM\n(100 LM, 30 particles)', 'EKF-SLAM\n(100 LM)'],
              [t_fast_l * 1000, t_ekf_est * 1000],
              color=['orange', 'steelblue'], alpha=0.8)
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title('Computation time: 100 landmarks', fontsize=13)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{height:.1f} ms', ha='center', fontsize=12)
plt.tight_layout()
plt.show()

best_l = max(fast_large, key=lambda p: p.weight)
n_seen = sum(1 for j in range(N_LM_large) if best_l.lm_seen[j])
print(f'FastSLAM: {N_LM_large} landmarks, {n_seen} observed, {t_fast_l*1000:.1f} ms total')
print(f'EKF-SLAM: {N_LM_large} landmarks, {t_ekf_est*1000:.1f} ms for updates alone')

**Capstone observations:**
- For 10 landmarks, both algorithms produce good maps. EKF-SLAM may be slightly more accurate because it maintains exact correlations.
- For 100 landmarks, FastSLAM remains practical while EKF-SLAM's covariance operations become expensive.
- FastSLAM's accuracy depends on particle count. More particles = better approximation of the posterior, but higher cost.
- Particle depletion (all weight concentrated on few particles) is the main failure mode of FastSLAM.

---

## Exercises

### Exercise 22.1: Vary the particle count

Run FastSLAM with 5, 20, 50, and 200 particles on the 10-landmark problem.
Plot mean landmark error vs number of particles. How many particles are
"enough"?

In [ ]:
# Your code here

### Exercise 22.2: Particle depletion

Run FastSLAM with only 5 particles on a long trajectory (80 steps).
Track the number of unique particles after resampling at each step.
What happens when all particles collapse to a single trajectory?

In [ ]:
# Your code here

### Exercise 22.3: Compare maps (challenge)

Run both EKF-SLAM and FastSLAM on the exact same set of observations
(same random seed, same measurements). Compare their final landmark
estimates side by side. Which one is closer to truth? Why?

In [ ]:
# Your code here